# Notebook 10 — LLM-as-judge (OpenAI **ou** Anthropic)

Évalue les réponses sur **1–5** : **cohérence**, **utilité**, **fidélité** au contexte de `test.json`.

**Entrées** : un fichier `*_predictions.json` + `data/processed/test.json` (jointure par `pair_id`).

**Sortie** : `results/llm_judge_<étiquette>.json` (résumé + détail).

**Clé** : `OPENAI_API_KEY` (provider OpenAI) ou `ANTHROPIC_API_KEY` (provider Anthropic).
Modèles conseillés : `gpt-4o` (OpenAI) ou `claude-sonnet-4-6` (Anthropic).

Si `scripts/llm_judge.py` est présent sur le Drive, il est chargé ; **sinon le code du juge est exécuté en mode intégré** (aucune copie de fichier nécessaire).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/llm-integration-study/" 


In [ ]:
!pip install -q openai anthropic


In [ ]:
import os, sys, json, time, getpass, importlib.util, types

from openai import OpenAI
from anthropic import Anthropic

PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")
RESULTS_PATH   = os.path.join(BASE_PATH, "results")


def _embed_llm_judge():
    """Même logique que scripts/llm_judge.py si le fichier n'est pas sur le Drive."""
    DEFAULT_MODEL = "gpt-4o"
    JUDGE_SYSTEM = """Tu es un évaluateur neutre pour des réponses en français.
Tu dois renvoyer UNIQUEMENT un objet JSON valide avec les clés :
  "coherence" : entier de 1 à 5 (cohérence interne et clarté de la réponse),
  "utilite"   : entier de 1 à 5 (à quel point la réponse répond utilement à la question),
  "fidelite"  : entier de 1 à 5 (respect du contexte fourni ; pénalise les inventions contraires au contexte),
  "commentaire_bref" : chaîne courte (optionnelle, une phrase).

Échelle : 1 = très mauvais, 3 = acceptable, 5 = excellent.
Les trois scores doivent être des entiers entre 1 et 5 inclus."""

    def build_user_content(question, context, reference_answer, predicted):
        ctx = (context or "").strip()
        if len(ctx) > 8000:
            ctx = ctx[:8000] + "\n[… contexte tronqué …]"
        ref = (reference_answer or "").strip()
        pred = (predicted or "").strip()
        q = (question or "").strip()
        return f"""Question :
{q}

Contexte de référence (source documentaire ; sert à juger la fidélité) :
{ctx}

Réponse de référence (résumé attendu, indication de qualité) :
{ref}

Réponse du système à évaluer :
{pred}

Évalue uniquement la « Réponse du système ». Renvoie le JSON demandé."""

    def judge_one(client, *, question, context, reference_answer, predicted, model=DEFAULT_MODEL, temperature=0.0):
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM},
                {"role": "user", "content": build_user_content(question, context, reference_answer, predicted)},
            ],
            temperature=temperature,
            response_format={"type": "json_object"},
        )
        text = (resp.choices[0].message.content or "").strip()
        try:
            out = json.loads(text)
        except json.JSONDecodeError:
            return {"coherence": None, "utilite": None, "fidelite": None, "commentaire_bref": "parse_error", "raw": text}
        for k in ("coherence", "utilite", "fidelite"):
            if k in out and out[k] is not None:
                try:
                    out[k] = int(out[k])
                except (TypeError, ValueError):
                    pass
        return out

    def load_test_index(path):
        with open(path, "r", encoding="utf-8") as f:
            rows = json.load(f)
        by_id = {}
        for row in rows:
            pid = row.get("pair_id") or row.get("id")
            if pid:
                by_id[str(pid)] = row
        return by_id

    def _mean_int(rows, key):
        vals = []
        for r in rows:
            v = r.get(key)
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                vals.append(float(v))
        return None if not vals else sum(vals) / len(vals)

    def run_judge_on_predictions(
        *,
        predictions_path,
        test_json_path,
        out_path,
        model=DEFAULT_MODEL,
        api_key=None,
        max_items=None,
        sleep_s=0.3,
        predictions_label="",
    ):
        key = api_key or os.environ.get("OPENAI_API_KEY")
        if not key:
            raise ValueError("Clé OpenAI manquante")
        client = OpenAI(api_key=key)
        with open(predictions_path, "r", encoding="utf-8") as f:
            preds = json.load(f)
        test_by_id = load_test_index(test_json_path)
        results = []
        for i, p in enumerate(preds):
            if max_items is not None and i >= max_items:
                break
            pid = str(p.get("pair_id", ""))
            t = test_by_id.get(pid, {})
            question = p.get("question") or t.get("question", "")
            predicted = p.get("predicted_answer", "")
            ref = p.get("true_answer") or t.get("answer", "")
            ctx = t.get("context", "")
            scores = judge_one(client, question=question, context=ctx, reference_answer=ref, predicted=predicted, model=model)
            results.append({
                "pair_id": pid,
                "predictions_file": predictions_path,
                "predictions_label": predictions_label or os.path.basename(predictions_path),
                "model_judge": model,
                **scores,
            })
            time.sleep(sleep_s)
        summary = {
            "predictions_file": predictions_path,
            "test_file": test_json_path,
            "judge_model": model,
            "n": len(results),
            "mean_coherence": _mean_int(results, "coherence"),
            "mean_utilite": _mean_int(results, "utilite"),
            "mean_fidelite": _mean_int(results, "fidelite"),
        }
        payload = {"summary": summary, "details": results}
        d = os.path.dirname(out_path)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        return results

    m = types.SimpleNamespace()
    m.run_judge_on_predictions = run_judge_on_predictions
    m.DEFAULT_MODEL = DEFAULT_MODEL
    return m


def _load_llm_judge():
    candidates = [
        os.path.join(BASE_PATH, "scripts", "llm_judge.py"),
        os.path.join(BASE_PATH, "Article_scientifique", "scripts", "llm_judge.py"),
        "/content/drive/MyDrive/Idee_random/Article_scientifique/scripts/llm_judge.py",
        os.path.join(os.getcwd(), "scripts", "llm_judge.py"),
        os.path.join(os.getcwd(), "Article_scientifique", "scripts", "llm_judge.py"),
    ]
    for p in candidates:
        if os.path.isfile(p):
            spec = importlib.util.spec_from_file_location("llm_judge", p)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            print(f"[llm_judge] chargé depuis : {p}")
            return mod
    print("[llm_judge] fichier absent du Drive → utilisation du module intégré.")
    return _embed_llm_judge()


judge_mod = _load_llm_judge()


def _run_judge_on_predictions_anthropic(
    *,
    predictions_path,
    test_json_path,
    out_path,
    model="claude-sonnet-4-6",
    api_key=None,
    max_items=None,
    sleep_s=0.35,
    predictions_label="",
):
    key = api_key or os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise ValueError("Clé Anthropic manquante")
    client = Anthropic(api_key=key)

    with open(predictions_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    with open(test_json_path, "r", encoding="utf-8") as f:
        test_rows = json.load(f)
    test_by_id = {str(r.get("pair_id") or r.get("id") or ""): r for r in test_rows}

    system_prompt = (
        "Tu es un évaluateur neutre pour des réponses en français. "
        "Tu dois renvoyer UNIQUEMENT un JSON valide avec les clés: coherence (1-5), utilite (1-5), "
        "fidelite (1-5), commentaire_bref (optionnel)."
    )

    def _safe_json(text):
        try:
            out = json.loads((text or "").strip())
        except Exception:
            out = {"coherence": None, "utilite": None, "fidelite": None, "commentaire_bref": "parse_error", "raw": text}
        for k in ("coherence", "utilite", "fidelite"):
            if k in out and out[k] is not None:
                try:
                    out[k] = int(out[k])
                except Exception:
                    pass
        return out

    rows = []
    for i, p in enumerate(preds):
        if max_items is not None and i >= max_items:
            break
        pid = str(p.get("pair_id", ""))
        t = test_by_id.get(pid, {})
        q = p.get("question") or t.get("question", "")
        pred = p.get("predicted_answer", "")
        ref = p.get("true_answer") or t.get("answer", "")
        ctx = t.get("context", "")
        user_prompt = f"""Question :
{q}

Contexte de référence :
{ctx}

Réponse de référence :
{ref}

Réponse du système à évaluer :
{pred}

Rends UNIQUEMENT un JSON: {{\"coherence\":1-5,\"utilite\":1-5,\"fidelite\":1-5,\"commentaire_bref\":\"...\"}}"""
        r = client.messages.create(
            model=model,
            max_tokens=512,
            temperature=0.0,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
        )
        txt = "\n".join(getattr(b, "text", "") for b in (getattr(r, "content", []) or []) if getattr(b, "text", None))
        scores = _safe_json(txt)
        rows.append({
            "pair_id": pid,
            "predictions_file": predictions_path,
            "predictions_label": predictions_label or os.path.basename(predictions_path),
            "model_judge": model,
            **scores,
        })
        time.sleep(sleep_s)

    def _mean_int(items, key):
        vals = [float(x[key]) for x in items if isinstance(x.get(key), (int, float)) and not isinstance(x.get(key), bool)]
        return None if not vals else sum(vals) / len(vals)

    payload = {
        "summary": {
            "predictions_file": predictions_path,
            "test_file": test_json_path,
            "judge_model": model,
            "n": len(rows),
            "mean_coherence": _mean_int(rows, "coherence"),
            "mean_utilite": _mean_int(rows, "utilite"),
            "mean_fidelite": _mean_int(rows, "fidelite"),
        },
        "details": rows,
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


PREDICTIONS_FILE = os.path.join(RESULTS_PATH, "rag_predictions.json")
JUDGE_PROVIDER   = "anthropic"  # "openai" ou "anthropic"
JUDGE_MODEL      = "claude-sonnet-4-6" if JUDGE_PROVIDER == "anthropic" else "gpt-4o"
OUT_FILE         = os.path.join(RESULTS_PATH, f"llm_judge_{JUDGE_PROVIDER}_rag.json")
MAX_ITEMS        = None

if JUDGE_PROVIDER == "anthropic":
    key = os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("ANTHROPIC_API_KEY : ")
    os.environ["ANTHROPIC_API_KEY"] = key
    _run_judge_on_predictions_anthropic(
        predictions_path=PREDICTIONS_FILE,
        test_json_path=os.path.join(PROCESSED_PATH, "test.json"),
        out_path=OUT_FILE,
        model=JUDGE_MODEL,
        max_items=MAX_ITEMS,
        sleep_s=0.35,
        predictions_label=os.path.basename(PREDICTIONS_FILE),
    )
else:
    key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OPENAI_API_KEY : ")
    os.environ["OPENAI_API_KEY"] = key
    judge_mod.run_judge_on_predictions(
        predictions_path=PREDICTIONS_FILE,
        test_json_path=os.path.join(PROCESSED_PATH, "test.json"),
        out_path=OUT_FILE,
        model=JUDGE_MODEL,
        max_items=MAX_ITEMS,
        sleep_s=0.35,
        predictions_label=os.path.basename(PREDICTIONS_FILE),
    )

print("OK →", OUT_FILE)
with open(OUT_FILE, "r", encoding="utf-8") as f:
    pack = json.load(f)
print("Moyennes :", json.dumps(pack["summary"], ensure_ascii=False, indent=2))
